<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Elena/xlm-roberta-large-xnli-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

Cloning into 'DataScienceCapstoneProject'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 176 (delta 65), reused 20 (delta 20), pack-reused 88 (from 1)
Receiving objects: 100% (176/176), 1.52 MiB | 11.42 MiB/s, done.
Resolving deltas: 100% (98/98), done.


 # 1. Zero-shot classification using a pretrained NLI model

In [ ]:
from transformers import pipeline

# Try another model for comparison:"joeddav/xlm-roberta-large-xnli"
# This model takes xlm-roberta-large and fine-tunes it on a combination of NLI data in 15 languages. It is intended to be used for zero-shot text classification, such as with the Hugging Face ZeroShotClassificationPipeline.
# This model is intended to be used for zero-shot text classification, especially in languages other than English. It is fine-tuned on XNLI, which is a multilingual NLI dataset.

pipe = pipeline("zero-shot-classification",model="joeddav/xlm-roberta-large-xnli")


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cuda:0


We load a zero-shot classification pipeline from Hugging Face.

The model joeddav/xlm-roberta-large-xnli is:

  * Based on XLM-RoBERTa-large
  * Fine-tuned on XNLI (Cross-lingual Natural Language Inference) data
  * Designed to classify text without task-specific training

Zero-shot classification works by reframing classification as an NLI task:
  * Premise: the input text (job position)
  * Hypothesis: “This text belongs to category X”

This makes the model suitable when labeled training data is limited or unavailable.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/DataScienceCapstoneProject/df_profiles_cleansed.csv")
# Predict department with a pretrained model based on position name
dpt_labels = df["department"].drop_duplicates()
text = df["position"].tolist()
candidate_labels = dpt_labels
result = pipe(text, candidate_labels)

* candidate_labels are all unique department names in the dataset.
* Each job position is classified against all department labels.
* The model returns a ranked list of labels for each position with confidence scores.

The dataset contains job profiles with fields such as:
position, department, seniority

# 2. Zero-shot prediction of department

In [ ]:
prediction = []
# For each position, we take the top-ranked predicted department.
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

# Evaluation
from sklearn.metrics import classification_report
print(classification_report(df["department"], prediction))

                        precision    recall  f1-score   support

        Administrative       0.15      0.54      0.23        84
  Business Development       0.49      0.51      0.50        78
            Consulting       0.46      0.62      0.53       195
      Customer Support       0.62      0.60      0.61        48
       Human Resources       0.30      0.91      0.45        69
Information Technology       0.69      0.63      0.66       309
             Marketing       0.58      0.40      0.47       133
                 Other       0.73      0.53      0.61      1235
    Project Management       0.74      0.68      0.70       173
            Purchasing       0.32      0.60      0.42        72
                 Sales       0.79      0.51      0.62       219

              accuracy                           0.56      2615
             macro avg       0.53      0.59      0.53      2615
          weighted avg       0.65      0.56      0.59      2615



# Result interpretation (Department)

## Accuracy = 56%
xlm-roberta-large-xnli captures semantic meaning of job titles.
Especially useful in multilingual or non-standard job naming contexts.

# 3. Zero-shot prediction of seniority


In [ ]:
sen_labels = df["seniority"].drop_duplicates()
text = df["position"].tolist()
candidate_labels = sen_labels
result = pipe(text, candidate_labels)

# The same zero-shot approach is applied.
# Only the label space changes from departments → seniority levels.

prediction = []
for i in range(len(result)):
  prediction.append(result[i]["labels"][0])

from sklearn.metrics import classification_report
print(classification_report(df["seniority"], prediction))

              precision    recall  f1-score   support

    Director       0.48      0.78      0.60       141
      Junior       0.84      0.54      0.66       230
        Lead       0.54      0.29      0.38       455
  Management       0.05      0.03      0.04       408
Professional       0.71      0.73      0.72      1211
      Senior       0.16      0.52      0.25       170

    accuracy                           0.52      2615
   macro avg       0.47      0.48      0.44      2615
weighted avg       0.54      0.52      0.51      2615



# Result interpretation (Seniority)
## Accuracy = 52%

Seniority terms (e.g., Junior, Lead, Manager) are:
* More subtle
* Often ambiguous in job titles

Zero-shot models struggle when label distinctions are not explicit in text.

# 4. Attempt to fine-tune the model on labeled data

xlm-roberta-large-xnli showed acceptable overall zero-shot performance, and we attempted to:

* Fine-tune it using 80% labeled data in CV dataset
* Evaluate on 20% held-out data
* Expect improved task-specific accuracy

## Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Departments are converted from strings → numeric labels
# Required for supervised training
le = LabelEncoder()
labels = le.fit_transform(df["department"].drop_duplicates())
dpt = le.inverse_transform([ 7,  5,  3,  2,  8, 10,  1,  0,  6,  4,  9])
mapping = dict(zip(dpt, labels))
dataset = df[["position", "department"]]
dataset["labels"] = df["department"].map(mapping)
dataset = dataset.drop(columns = "department")

/tmp/ipython-input-2939481628.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset["labels"] = df["department"].map(mapping)


## Train-test split

In [ ]:
from datasets import Dataset
hf_dataset = Dataset.from_pandas(dataset)
split_datasets = hf_dataset.train_test_split(test_size = 0.2, seed = 42)
train_dataset = split_datasets["train"]
eval_dataset = split_datasets["test"]

## Tokenization

In [ ]:
from transformers import AutoTokenizer

model_name = "joeddav/xlm-roberta-large-xnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Converts text into:input_ids, attention_mask
# max_length=106 is chosen based on job title length

def tokenize(batch):
  return tokenizer(batch["position"], truncation=True, padding = "max_length", max_length = 106)

train_dataset = train_dataset.map(tokenize, batched = True)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

eval_dataset = train_dataset.map(tokenize, batched = True)
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/2092 [00:00<?, ? examples/s]

Map:   0%|          | 0/2092 [00:00<?, ? examples/s]

In [ ]:
train_dataset

Dataset({
    features: ['position', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 2092
})

# 5. Model initialization for supervised classification

In [ ]:
 id2label = {int(k):str(v) for k,v in zip(labels, dpt)}
 label2id = {str(v):int(k) for k,v in id2label.items()}

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(label2id),
    id2label=id2label,
    label2id = label2id,
    ignore_mismatched_sizes = True
)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at joeddav/xlm-roberta-large-xnli and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([11]) in the

* We redefine the classification head to predict departments
* ignore_mismatched_sizes=True forces reinitialization of the output layer
* Consequence: The original NLI classification head (entailment / neutral / contradiction) is discarded

# 6. Training and evaluation

In [ ]:
# Train the model
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(
    "test_trainer", report_to="none")

In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
  logits,labels = eval_pred
  predictions = np.argmax(logits, axis = -1)
  return metric.compute(predictions=predictions, references=labels)

In [ ]:
trainer = Trainer(
  model = model,
  args = training_args,
  train_dataset = train_dataset,
  eval_dataset = eval_dataset,
  compute_metrics = compute_metrics
)

trainer.evaluate()

{'eval_loss': 2.354811906814575,
 'eval_model_preparation_time': 0.0547,
 'eval_accuracy': 0.11424474187380497,
 'eval_runtime': 44.716,
 'eval_samples_per_second': 46.784,
 'eval_steps_per_second': 5.859}

## Result interpretation (after training)
## Accuracy = 11%

This is much worse than the 56% zero-shot accuracy.

# 7. Why performance collapses after training

After traing, our model accuracy on evaluation data comes down to only 11%, compared to 56% before training. It seems that our model is broken after training.
Here are the possible reasons :

### Reason 1
xlm-roberta-large-xnli is not a standard classifier. It is trained for Natural Language Inference (NLI):
premise + hypothesis → {entailment, neutral, contradiction}
Zero-shot classification works because Hugging Face reformulates classification as NLI:
*   Premise: "Senior Data Analyst"
*   Hypothesis: "This job is in the Finance department."

When we fine-tuned it directly on:
*   input: job_position
*   label: department
The NLI alignment that makes zero-shot work is destroyed.

### Reason 2
We trained with num_labels = number_of_departments.But the pretrained model expects 3 NLI labels.This silently breaks the pretrained head.The pretrained classification head is reinitialized.
All zero-shot knowledge is lost.
We are training a random classifier from scratch.

### Reason 3
20% of CV data is probably far too small.

# Final conclusion

* Zero-shot classification with xlm-roberta-large-xnli is: Effective
, Data-efficient, Suitable for this task.

* Fine-tuning this model directly is not appropriate.
It is not a standard classifier
Its strength lies in NLI-based zero-shot inference.

* For supervised learning, a better approach would be:
A model pretrained for sequence classification.
Or reformulating training data explicitly as NLI pairs